@WJR: MS Video zum Thema Vector Suche: https://www.youtube.com/watch?v=otlYeBH9U6E

In [ ]:
%pip install openai python-dotenv

In [7]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import numpy as np
load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )

model = 'text-embedding-ada-002'

In [ ]:
# Dependencies for embeddings_utils
%pip install matplotlib plotly scikit-learn pandas

In [2]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [4]:
# compare several words
automobile_embedding    = client.embeddings.create(input = 'automobile', model=model).data[0].embedding
vehicle_embedding       = client.embeddings.create(input = 'vehicle', model=model).data[0].embedding
dinosaur_embedding      = client.embeddings.create(input = 'dinosaur', model=model).data[0].embedding
stick_embedding         = client.embeddings.create(input = 'stick', model=model).data[0].embedding

# comparing cosine similarity, automobiles vs automobiles should be 1.0, i.e exactly the same, while automobiles vs dinosaurs should be between 0 and 1, i.e. not the same
print(cosine_similarity(automobile_embedding, automobile_embedding))
print(cosine_similarity(automobile_embedding, vehicle_embedding))
print(cosine_similarity(automobile_embedding, dinosaur_embedding))
print(cosine_similarity(automobile_embedding, stick_embedding))

1.0
0.9157394164180288
0.8329017718558469
0.7818148774886837


In [19]:
import pandas as pd

AI_SHOW_TRANSCRIPT_EMBEDDINGS_PATH = '../embedding_index_3m.json'

ai_show_transcript_embeddings = pd.read_json(AI_SHOW_TRANSCRIPT_EMBEDDINGS_PATH)

ai_show_transcript_embeddings.head(2)

,speaker,title,videoId,start,seconds,summary,ada_v2
0,"Seth Juarez, Josh Lovejoy, Sarah Bird",You're Not Solving the Problem You Think You'r...,-tJQm4mSh1s,00:00:00,0,Join Seth Juarez as he discusses ethical conce...,"[0.004357332363724, -0.028409153223037, 0.0111..."
1,"Seth Juarez, Josh Lovejoy, Sarah Bird",You're Not Solving the Problem You Think You'r...,-tJQm4mSh1s,00:03:07,187,"In this video, the speaker discusses the chall...","[-0.0038613036740570003, -0.004626247566193000..."


In [59]:
def get_embedding(question):
    return client.embeddings.create(input = question, model=model).data[0].embedding

def get_closest_entries(embedding, top_n=5):
    similarity_ai_show_transcript_embeddings = ai_show_transcript_embeddings.copy()
    similarity_ai_show_transcript_embeddings['similarity'] = similarity_ai_show_transcript_embeddings.apply(lambda row: cosine_similarity(row['ada_v2'], embedding), axis=1)
    return similarity_ai_show_transcript_embeddings.sort_values(by='similarity', ascending=False).head(top_n)
    
    

In [62]:
question = input("What do you want to know about ai?: ")
top_n = 5
question_embedding = get_embedding(question)
most_similar = get_closest_entries(question_embedding, top_n)
print(r'---------------------- Top {top_n} videos ------------------------')
print(most_similar[['videoId', 'similarity']])

What do you want to know about ai?:  All about NLP


---------------------- Top {top_n} videos ------------------------
          videoId  similarity
1342  vwSYCy-NLqU    0.850276
859   c0eZJ95ZVA4    0.839856
1067  mYKGm5LeqFg    0.832810
597   Rdm5Xjq6HmQ    0.831032
1345  vwSYCy-NLqU    0.828674
